# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [1]:
# Code here
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.impute import KNNImputer

df = pd.read_csv("./Income Dirty Data.csv")

3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [ ]:
# Total # of Nan values for each col
print(df.isna().sum())

# % of Nan
percent = (df.isna().sum().sum()/df.size) *100
print("\nPercentage of Nan values in dataset")
print(f"{percent:.2f}%")


ID              0
sex            88
age             0
income        109
tax_15_pct     93
dtype: int64

Percentage of Nan values in dataset
5.80%


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [ ]:
# 4. % of data that follows the rules
valid = (df['age'] >= 18) & (df['income'] > 0) & (np.isclose(df['tax_15_pct'], df['income'] * 0.15))
valid_percent = valid.mean() * 100
print(f"{valid_percent:.2f}% of rows follow all rules")

59.80% of rows follow all rules


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [12]:
gender_map = {
    "Men" : "Male", "Man":"Male",
    "Woman":"Female", "Women":"Female"
}

df["sex"] = df["sex"].replace(gender_map)
df.head(20)

,ID,sex,age,income,tax_15_pct
0,1,Female,21,147168.0,22075.20
1,2,Female,29,119595.0,17939.25
2,3,Female,56,87770.0,13165.50
3,4,NaN,21,54259.0,8138.85
4,5,Male,28,NaN,160230.00
5,6,Female,0,128326.0,19248.90
6,7,Female,-1,NaN,NaN
7,8,Female,24,NaN,11820.60
8,9,Female,38,149473.0,22420.95
9,10,Male,48,113663.0,1136630.00


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [14]:
df.loc[df['age'] <= 0, 'age'] = np.nan
df.loc[df['income'] <= 0, 'income'] = np.nan
df.loc[df['tax_15_pct'] <= 0, 'tax_15_pct'] = np.nan

print(df.head(20))

    ID     sex   age    income  tax_15_pct
0    1  Female  21.0  147168.0    22075.20
1    2  Female  29.0  119595.0    17939.25
2    3  Female  56.0   87770.0    13165.50
3    4     NaN  21.0   54259.0     8138.85
4    5    Male  28.0       NaN   160230.00
5    6  Female   NaN  128326.0    19248.90
6    7  Female   NaN       NaN         NaN
7    8  Female  24.0       NaN    11820.60
8    9  Female  38.0  149473.0    22420.95
9   10    Male  48.0  113663.0  1136630.00
10  11  Female  33.0   96649.0    14497.35
11  12    Male  55.0       NaN     9776.25
12  13    Male  52.0   64944.0     9741.60
13  14    Male  47.0   85300.0         NaN
14  15    Male  45.0   80091.0    12013.65
15  16  Female  25.0   56418.0     8462.70
16  17  Female  45.0   54189.0     8128.35
17  18    Male  29.0   67489.0    10123.35
18  19    Male  48.0  137978.0    20696.70
19  20  Female   NaN   89991.0         NaN


The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [ ]:
# 9. KNN imputation
print("Before imputing:")
print(df.head())

df_imputed = df.copy()
df_imputed['sex'] = df_imputed['sex'].map({'Female': 0, 'Male': 1})

scaler = preprocessing.StandardScaler()
scaled = scaler.fit_transform(df_imputed)

imputer = KNNImputer(n_neighbors=5)
imputed_scaled = imputer.fit_transform(scaled)

df_imputed = pd.DataFrame(scaler.inverse_transform(imputed_scaled), columns=df_imputed.columns)
df_imputed['sex'] = df_imputed['sex'].round().map({0: 'Female', 1: 'Male'})

print("\nAfter imputing:")
print(df_imputed.head())

Before imputing:
   ID     sex   age    income  tax_15_pct
0   1  Female  21.0  147168.0    22075.20
1   2  Female  29.0  119595.0    17939.25
2   3  Female  56.0   87770.0    13165.50
3   4     NaN  21.0   54259.0     8138.85
4   5    Male  28.0       NaN   160230.00

After imputing:
    ID     sex   age    income  tax_15_pct
0  1.0  Female  21.0  147168.0    22075.20
1  2.0  Female  29.0  119595.0    17939.25
2  3.0  Female  56.0   87770.0    13165.50
3  4.0    Male  21.0   54259.0     8138.85
4  5.0    Male  28.0  114375.4   160230.00


10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data

Cleaning data before analysis matters because errors, missing values, and inconsistent formatting skew statistics like means and correlations, leading to inaccurate or misleading conclusions. Analytics run on dirty data can produce results that look valid but don't reflect reality, causing bad decisions downstream.


### Submission to D2L Dropbox
- Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment5.ipynb** 
    - Replace '**Last**' and '**First**' with your first and last name

- Include the link to your GitHub repository (the URL of your repo page, for
   example `https://github.com/yourname/csci-4047-work`).
### How to add my code to GitHub?

1. Stage your file (this tells Git which changes to include):

      `git add Last_First_Assignment5.ipynb`

   To stage everything (you might not want to stage everything though) in the folder instead, use `git add .`

3. Commit your changes (this saves a snapshot with a message):

       git commit -m "Add Assignment 5"

   The text in quotes is your commit message. Make it describe what you
   did.

4. Push your commit up to GitHub:

       git push -u origin main

   The `-u origin main` part is only needed the first push. After that,
   `git push` alone is enough.

**What each command does, briefly**

- `git add` picks which files to include in the next save.
- `git commit` saves a snapshot of those files on your computer, with a
  message describing the change.
- `git push` uploads your saved commits to GitHub so they appear online.